# Clinical Clustering Analysis

In [ ]:
# Install required packages
!pip install -q scikit-learn scipy pandas matplotlib seaborn umap-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, 
    davies_bouldin_score, 
    calinski_harabasz_score,
    silhouette_samples
)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from scipy.stats import kruskal, mannwhitneyu
import umap

In [ ]:
# Load dataset
notebook_dir = Path().resolve()
FEATURE_PATH = notebook_dir.parents[1] / "1" / "Features"
PATIENT_PROFILES_NAME = "patient_profiles.csv"

df = pd.read_csv(FEATURE_PATH / PATIENT_PROFILES_NAME)

print(f"Dataset loaded: {df.shape[0]} patients, {df.shape[1]} features")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nFeature columns:\n{df.columns.tolist()}")

## Feature Organization

### Feature Groups

In [ ]:
# Define feature groups for clinical interpretation
feature_groups = {
    "Demographics": [
        "age",
        "gender"
    ],
    
    "Vital Signs": [
        "temperature",
        "heart_rate",
        "respiratory_rate",
        "oxygen_saturation",
        "bp_systolic",
    ],
    
    "Lab Values - Electrolytes": [
        "k_min",
        "k_max",
    ],
    
    "Lab Values - Metabolic & Renal": [
        "metabolic_dysregulation",
        "renal_function_category",
    ],
    
    "Clinical Severity Scores": [
        "disease_burden_score",
    ],
    
    "Healthcare Utilization": [
        "total_procedures",
        "procedure_complexity",
        "total_tests",
        "testing_intensity",
    ],
    
    "Risk Indicators": [
        "high_risk_patient",
        "complex_patient",
    ],
    
    "Missingness Indicators": [
        "gender_missing",
        "cad_severity_score_missing",
        "ct_complications_score_missing",
        "infection_severity_score_missing",
    ],
}

# Flatten feature groups for clustering
all_features = []
for group, features in feature_groups.items():
    all_features.extend(features)

print("Feature Groups Summary:")
print("=" * 60)
for group, features in feature_groups.items():
    print(f"{group:40s}: {len(features):2d} features")
print("=" * 60)
print(f"{'Total':40s}: {len(all_features):2d} features")

In [ ]:
# Validate that all features exist in dataframe
missing_features = [f for f in all_features if f not in df.columns]
if missing_features:
    print(f"Missing features: {missing_features}")
else:
    print("All features validated")

# Select features for clustering
features_for_clustering = df[all_features].copy()
print(f"\nClustering dataset: {features_for_clustering.shape}")

In [ ]:
# Summary statistics by feature group
for group, features in feature_groups.items():
    available_features = [f for f in features if f in df.columns]
    if available_features:
        print(f"\n{group}")
        print("-" * 80)
        display(df[available_features].describe().T)

In [ ]:
# Distribution plots for key continuous features
continuous_features = [
    'age', 'disease_burden_score', 'metabolic_dysregulation',
    'procedure_complexity', 'testing_intensity', 'total_procedures'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(continuous_features):
    if feature in df.columns:
        axes[idx].hist(df[feature].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'Distribution: {feature}')
        axes[idx].set_xlabel(feature)
        axes[idx].set_ylabel('Frequency')
        axes[idx].axvline(df[feature].mean(), color='red', linestyle='--', 
                         label=f'Mean: {df[feature].mean():.2f}')
        axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for clinical features
clinical_features = (
    feature_groups['Clinical Severity Scores'] + 
    feature_groups['Risk Indicators'] +
    ['age', 'metabolic_dysregulation', 'renal_function_category']
)

available_clinical = [f for f in clinical_features if f in df.columns]
correlation_matrix = df[available_clinical].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1)
plt.title('Correlation Matrix: Clinical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Data Preprocessing for Clustering

In [ ]:
# Separate binary and continuous features for appropriate scaling
from sklearn.compose import ColumnTransformer

binary_cols = []
continuous_cols = []

for col in features_for_clustering.columns:
    unique_vals = features_for_clustering[col].nunique()
    if unique_vals <= 2:
        binary_cols.append(col)
    else:
        continuous_cols.append(col)

print(f"Binary features: {len(binary_cols)}")
print(binary_cols)
print(f"\nContinuous features: {len(continuous_cols)}")
print(continuous_cols)

In [ ]:
# features_for_clustering = features_for_clustering[clinical_features]

In [ ]:
# Create preprocessing pipeline
# Binary: StandardScaler (to normalize 0/1 to similar scale as continuous)
# Continuous: RobustScaler (less sensitive to outliers)
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', StandardScaler(), binary_cols),
        ('continuous', RobustScaler(), continuous_cols)
    ]
)

X_scaled = preprocessor.fit_transform(features_for_clustering)
X_scaled_df = pd.DataFrame(
    X_scaled, 
    columns=binary_cols + continuous_cols,
    index=features_for_clustering.index
)

print("\n✅ Data scaled successfully")
print(f"Scaled data shape: {X_scaled_df.shape}")
print(f"\nScaled data statistics:")
print(f"Mean (should be ~0 for continuous): {X_scaled[:, len(binary_cols):].mean(axis=0).mean():.4f}")
print(f"Std (should be ~1): {X_scaled.std(axis=0).mean():.4f}")

## Dimensionality Reduction for Visualization (PCA and UMAP)

In [ ]:
# PCA for linear dimensionality reduction
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA Variance Explained:")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"  Total: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Visualize PCA
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=20)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA Projection of Patient Data')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# UMAP for non-linear dimensionality reduction
print("Computing UMAP projection (this may take a minute)...")

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42,
    metric='euclidean'
)

X_umap = reducer.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], alpha=0.3, s=20)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('UMAP Projection of Patient Data')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Optimal Number of Clusters

In [ ]:
# Elbow Method + Silhouette Analysis
K_range = range(2, 11)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
calinski_harabasz_scores = []

print("Testing different numbers of clusters...")

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled, labels))
    calinski_harabasz_scores.append(calinski_harabasz_score(X_scaled, labels))
    
    print(f"k={k}: Silhouette={silhouette_scores[-1]:.3f}, "
          f"Davies-Bouldin={davies_bouldin_scores[-1]:.3f}")

In [ ]:
# Plot clustering metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Elbow plot
axes[0, 0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Clusters (k)')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].set_title('Elbow Method')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette score (higher is better)
axes[0, 1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Number of Clusters (k)')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].set_title('Silhouette Score (Higher = Better)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Good threshold')
axes[0, 1].legend()

# Davies-Bouldin index (lower is better)
axes[1, 0].plot(K_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Clusters (k)')
axes[1, 0].set_ylabel('Davies-Bouldin Index')
axes[1, 0].set_title('Davies-Bouldin Index (Lower = Better)')
axes[1, 0].grid(True, alpha=0.3)

# Calinski-Harabasz index (higher is better)
axes[1, 1].plot(K_range, calinski_harabasz_scores, 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Number of Clusters (k)')
axes[1, 1].set_ylabel('Calinski-Harabasz Index')
axes[1, 1].set_title('Calinski-Harabasz Index (Higher = Better)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Recommendation
best_k = K_range[np.argmax(silhouette_scores)]

## KMeans Clustering

In [ ]:
# Perform KMeans clustering with optimal k
optimal_k = 3  # Adjust based on metrics above

print(f"Performing KMeans clustering with k={optimal_k}...")

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Add labels to dataframe
df['kmeans_cluster'] = kmeans_labels

# Cluster statistics
print(f"\nCluster distribution:")
print(df['kmeans_cluster'].value_counts().sort_index())

# Clustering metrics
sil_score = silhouette_score(X_scaled, kmeans_labels)
db_score = davies_bouldin_score(X_scaled, kmeans_labels)
ch_score = calinski_harabasz_score(X_scaled, kmeans_labels)

print(f"\nClustering Quality Metrics:")
print(f"  Silhouette Score: {sil_score:.3f}")
print(f"  Davies-Bouldin Index: {db_score:.3f}")
print(f"  Calinski-Harabasz Index: {ch_score:.1f}")

In [ ]:
# Visualize clusters in PCA space
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PCA visualization
for i in range(optimal_k):
    mask = kmeans_labels == i
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               label=f'Cluster {i} (n={mask.sum()})',
               alpha=0.6, s=30)

ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax1.set_title('KMeans Clusters - PCA Projection')
ax1.legend()
ax1.grid(True, alpha=0.3)

# UMAP visualization
for i in range(optimal_k):
    mask = kmeans_labels == i
    ax2.scatter(X_umap[mask, 0], X_umap[mask, 1],
               label=f'Cluster {i} (n={mask.sum()})',
               alpha=0.6, s=30)

ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
ax2.set_title('KMeans Clusters - UMAP Projection')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Clinical Interpretation of Clusters

### Feature Comparison Across Clusters

In [ ]:
# Helper function to create clinical summaries
def create_cluster_summary(df, cluster_col='kmeans_cluster', feature_groups=feature_groups):
    """
    Create detailed clinical summary for each cluster.
    """
    summaries = {}
    
    for cluster_id in sorted(df[cluster_col].unique()):
        cluster_data = df[df[cluster_col] == cluster_id]
        n_patients = len(cluster_data)
        
        summary = {
            'n_patients': n_patients,
            'percentage': (n_patients / len(df)) * 100,
            'features': {}
        }
        
        # Compute statistics for each feature group
        for group_name, features in feature_groups.items():
            available_features = [f for f in features if f in df.columns]
            if available_features:
                group_stats = cluster_data[available_features].describe().loc[['mean', '50%', 'std']]
                summary['features'][group_name] = group_stats.T
        
        summaries[f'Cluster {cluster_id}'] = summary
    
    return summaries

# Create summaries
cluster_summaries = create_cluster_summary(df)

# Display summaries
for cluster_name, summary in cluster_summaries.items():
    print("="*80)
    print(f"{cluster_name}")
    print("="*80)
    print(f"Patients: {summary['n_patients']} ({summary['percentage']:.1f}%)")
    print("\n")
    
    for group_name, stats in summary['features'].items():
        print(f"\n{group_name}")
        print("-"*80)
        display(stats)
    print("\n")

In [ ]:
cluster_means = df.groupby('kmeans_cluster')[all_features].mean()

# Normalize for better visualization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cluster_means_normalized = pd.DataFrame(
    scaler.fit_transform(cluster_means.T).T,
    columns=cluster_means.columns,
    index=cluster_means.index
)

plt.figure(figsize=(16, 8))
sns.heatmap(cluster_means_normalized.T, cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Normalized Value'}, 
            linewidths=0.5, linecolor='gray')
plt.title('Feature Profile by Cluster (Normalized)', fontsize=14, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### Statistical Testing: Feature Differences Between Clusters

In [ ]:
# Kruskal-Wallis test for each feature
# (non-parametric test for differences across multiple groups)

kruskal_results = []

for feature in all_features:
    groups = [df[df['kmeans_cluster'] == i][feature].dropna() 
              for i in range(optimal_k)]
    
    # Only test if all groups have data
    if all(len(g) > 0 for g in groups):
        stat, p_value = kruskal(*groups)
        kruskal_results.append({
            'Feature': feature,
            'H-statistic': stat,
            'p-value': p_value,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })

kruskal_df = pd.DataFrame(kruskal_results).sort_values('p-value')

print("\nFeatures with Significant Differences Across Clusters (p < 0.05):")
print("="*80)
display(kruskal_df[kruskal_df['Significant'] == 'Yes'].head(15))

print(f"\nTotal significant features: {(kruskal_df['Significant'] == 'Yes').sum()}/{len(all_features)}")

### Box Plots: Key Clinical Features by Cluster

In [ ]:
# Select most discriminative features for visualization
top_features = kruskal_df.head(12)['Feature'].tolist()

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    if feature in df.columns:
        df.boxplot(column=feature, by='kmeans_cluster', ax=axes[idx])
        axes[idx].set_title(f'{feature}')
        axes[idx].set_xlabel('Cluster')
        axes[idx].set_ylabel('Value')
        plt.sca(axes[idx])
        plt.xticks(rotation=0)

# Remove the automatic title
plt.suptitle('')
fig.suptitle('Top Discriminative Features by Cluster', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### Clinical Phenotype Labels

Based on the feature profiles, we can assign clinical phenotypes to each cluster:

In [ ]:
# Function to generate clinical phenotype labels
def generate_phenotype_labels(df, cluster_col='kmeans_cluster'):
    """
    Generate clinical phenotype labels based on cluster characteristics.
    This is a template - adjust based on your actual cluster profiles.
    """
    phenotypes = {}
    
    for cluster_id in sorted(df[cluster_col].unique()):
        cluster_data = df[df[cluster_col] == cluster_id]
        
        # Calculate key indicators
        avg_age = cluster_data['age'].mean()
        avg_burden = cluster_data['disease_burden_score'].mean() if 'disease_burden_score' in df.columns else 0
        avg_procedures = cluster_data['total_procedures'].mean() if 'total_procedures' in df.columns else 0
        high_risk_pct = (cluster_data['high_risk_patient'].mean() * 100) if 'high_risk_patient' in df.columns else 0
        
        # Heuristic labeling (adjust based on your data)
        if avg_burden > df['disease_burden_score'].quantile(0.75) and high_risk_pct > 50:
            label = "High Acuity / Critical"
        elif avg_procedures > df['total_procedures'].quantile(0.75):
            label = "High Complexity / Intervention-Heavy"
        elif avg_age < 50 and avg_burden < df['disease_burden_score'].quantile(0.5):
            label = "Young / Low Complexity"
        elif avg_age > 70:
            label = "Elderly / Chronic Conditions"
        else:
            label = "Moderate Complexity"
        
        phenotypes[cluster_id] = {
            'label': label,
            'avg_age': avg_age,
            'avg_burden': avg_burden,
            'avg_procedures': avg_procedures,
            'high_risk_pct': high_risk_pct,
            'n_patients': len(cluster_data)
        }
    
    return phenotypes

# Generate phenotypes
phenotypes = generate_phenotype_labels(df)

print("Clinical Phenotypes by Cluster")
print("="*80)
for cluster_id, info in phenotypes.items():
    print(f"\nCluster {cluster_id}: {info['label']}")
    print(f"  Patients: {info['n_patients']}")
    print(f"  Avg Age: {info['avg_age']:.1f}")
    print(f"  Avg Disease Burden: {info['avg_burden']:.2f}")
    print(f"  Avg Procedures: {info['avg_procedures']:.1f}")
    print(f"  High Risk %: {info['high_risk_pct']:.1f}%")

## Hierarchical Clustering

In [ ]:
# Dendrogram for hierarchical clustering
# Use a sample for computational efficiency
sample_size = min(1000, len(df))
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

print(f"Computing hierarchical clustering on {sample_size} samples...")

linkage_matrix = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 8))
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=30,
    show_leaf_counts=True,
    leaf_font_size=10
)
plt.title('Hierarchical Clustering Dendrogram (Truncated)', fontsize=14, fontweight='bold')
plt.xlabel('Cluster Size')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
# Perform hierarchical clustering on full dataset
hierarchical = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)

df['hierarchical_cluster'] = hierarchical_labels

print(f"\nHierarchical Clustering Results:")
print(df['hierarchical_cluster'].value_counts().sort_index())

# Compare with KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(kmeans_labels, hierarchical_labels)
nmi = normalized_mutual_info_score(kmeans_labels, hierarchical_labels)

print(f"\nAgreement between KMeans and Hierarchical:")
print(f"  Adjusted Rand Index: {ari:.3f}")
print(f"  Normalized Mutual Information: {nmi:.3f}")

## DBSCAN: Density-Based Clustering

In [ ]:
# DBSCAN parameter tuning
# Test different epsilon values

from sklearn.neighbors import NearestNeighbors

# K-distance graph to find optimal eps
k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
distances, indices = nbrs.kneighbors(X_scaled)

# Sort and plot
distances = np.sort(distances[:, k-1], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(distances)
plt.ylabel(f'{k}-NN Distance')
plt.xlabel('Data Points sorted by distance')
plt.title('K-Distance Graph for DBSCAN Epsilon Selection')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Suggest epsilon (look for "elbow" in the plot)
suggested_eps = np.percentile(distances, 90)
print(f"\nSuggested epsilon (90th percentile): {suggested_eps:.3f}")

In [ ]:
# Perform DBSCAN
eps = 2.5  # Adjust based on k-distance graph
min_samples = 10

print(f"Running DBSCAN with eps={eps}, min_samples={min_samples}...")

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_scaled)

df['dbscan_cluster'] = dbscan_labels

# Statistics
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"\nDBSCAN Results:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of noise points: {n_noise} ({n_noise/len(df)*100:.1f}%)")
print(f"\nCluster distribution:")
print(df['dbscan_cluster'].value_counts().sort_index())

In [ ]:
# Visualize DBSCAN clusters
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PCA
unique_labels = set(dbscan_labels)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        color = 'gray'
        marker = 'x'
        label_str = 'Noise'
    else:
        marker = 'o'
        label_str = f'Cluster {label}'
    
    mask = dbscan_labels == label
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[color], marker=marker, label=f'{label_str} (n={mask.sum()})',
               alpha=0.6, s=30)

ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax1.set_title('DBSCAN Clusters - PCA Projection')
ax1.legend()
ax1.grid(True, alpha=0.3)

# UMAP
for label, color in zip(unique_labels, colors):
    if label == -1:
        color = 'gray'
        marker = 'x'
        label_str = 'Noise'
    else:
        marker = 'o'
        label_str = f'Cluster {label}'
    
    mask = dbscan_labels == label
    ax2.scatter(X_umap[mask, 0], X_umap[mask, 1],
               c=[color], marker=marker, label=f'{label_str} (n={mask.sum()})',
               alpha=0.6, s=30)

ax2.set_xlabel('UMAP 1')
ax2.set_ylabel('UMAP 2')
ax2.set_title('DBSCAN Clusters - UMAP Projection')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Model Comparison & Final Selection

In [ ]:
# Compare all clustering methods
comparison = pd.DataFrame({
    'Method': ['KMeans', 'Hierarchical', 'DBSCAN'],
    'N_Clusters': [
        len(df['kmeans_cluster'].unique()),
        len(df['hierarchical_cluster'].unique()),
        len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    ],
    'Silhouette': [
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, hierarchical_labels),
        silhouette_score(X_scaled, dbscan_labels) if len(set(dbscan_labels)) > 1 else np.nan
    ],
    'Davies-Bouldin': [
        davies_bouldin_score(X_scaled, kmeans_labels),
        davies_bouldin_score(X_scaled, hierarchical_labels),
        davies_bouldin_score(X_scaled, dbscan_labels) if len(set(dbscan_labels)) > 1 else np.nan
    ],
    'Calinski-Harabasz': [
        calinski_harabasz_score(X_scaled, kmeans_labels),
        calinski_harabasz_score(X_scaled, hierarchical_labels),
        calinski_harabasz_score(X_scaled, dbscan_labels) if len(set(dbscan_labels)) > 1 else np.nan
    ]
})

print("Clustering Method Comparison")
print("="*80)
display(comparison)

print("\n💡 Interpretation:")
print("  - Silhouette Score: Higher is better (>0.5 is good)")
print("  - Davies-Bouldin: Lower is better")
print("  - Calinski-Harabasz: Higher is better")